# Лабораторная работа №8: Оптимизация и деплой LLM в продакшен

## Цель работы
Изучить методы оптимизации больших языковых моделей для продакшена, включая квантование, ускорение инференса и развертывание через API.

## Теоретическая часть

### Проблемы продакшн-деплоя LLM
1. **Высокие требования к памяти** — большие модели требуют значительных ресурсов GPU
2. **Низкая скорость инференса** — генерация текста может быть медленной
3. **Масштабируемость** — необходимость обработки множества запросов одновременно
4. **Стоимость** — аренда GPU и инфраструктура могут быть дорогими

### Методы оптимизации

#### 1. Квантование
- **INT8/INT4** — уменьшение точности весов модели
- **QLoRA** — квантованное низкоранговое адаптирование
- Снижение потребления памяти в 2-4 раза с минимальной потерей качества

#### 2. Ускорение инференса
- **vLLM** — оптимизированный движок с PagedAttention
- **TensorRT-LLM** — оптимизация от NVIDIA
- **ONNX Runtime** — кроссплатформенное ускорение

#### 3. Специализированные серверы
- **TGI (Text Generation Inference)** — от Hugging Face
- **LMDeploy** — от OpenMMLab
- **llama.cpp** — эффективный инференс на CPU/GPU

## Практическая часть

### Установка зависимостей

In [ ]:
!pip install transformers accelerate bitsandbytes torch -q
!pip install fastapi uvicorn pydantic -q

### Импорт библиотек

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
import time
import os

# Для доступа к некоторым моделям требуется токен Hugging Face
# os.environ["HF_TOKEN"] = "your-token-here"

### Загрузка модели с квантованием

In [ ]:
# Конфигурация 4-битного квантования
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Модель для тестирования (небольшая модель для Colab)
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Загрузка токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Загрузка модели с 4-битным квантованием...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print(f"Модель загружена на устройство: {next(model.parameters()).device}")

### Сравнение производительности: обычная vs квантованная модель

In [ ]:
def measure_inference_time(model, tokenizer, prompt, max_new_tokens=50):
    """Измеряет время генерации текста."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Прогрев
    _ = model.generate(**inputs, max_new_tokens=10)
    
    # Измерение
    start_time = time.time()
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    end_time = time.time()
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    inference_time = end_time - start_time
    tokens_per_second = max_new_tokens / inference_time
    
    return generated_text, inference_time, tokens_per_second

test_prompt = "Расскажи кратко о преимуществах искусственного интеллекта."

print(f"Тестовый запрос: {test_prompt}\n")
print("Генерация ответа...")

response, inf_time, tps = measure_inference_time(model, tokenizer, test_prompt)

print(f"\nОтвет модели:\n{response}")
print(f"\nВремя инференса: {inf_time:.2f} сек")
print(f"Скорость: {tps:.2f} токенов/сек")
print(f"Использование памяти GPU: {torch.cuda.memory_allocated() / 1024**2:.2f} MB" if torch.cuda.is_available() else "CPU режим")

### Создание FastAPI сервера для деплоя

In [ ]:
%%writefile llm_server.py

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import uvicorn

# Модели данных для API
class GenerateRequest(BaseModel):
    prompt: str
    max_tokens: int = 100
    temperature: float = 0.7

class GenerateResponse(BaseModel):
    text: str
    inference_time: float
    tokens_per_second: float

# Инициализация приложения
app = FastAPI(title="LLM Inference API", version="1.0")

# Глобальные переменные для модели
model = None
tokenizer = None

@app.on_event("startup")
async def load_model():
    global model, tokenizer
    
    print("Загрузка модели...")
    model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    print("Модель успешно загружена!")

@app.post("/generate", response_model=GenerateResponse)
async def generate_text(request: GenerateRequest):
    if model is None or tokenizer is None:
        raise HTTPException(status_code=503, detail="Модель еще не загружена")
    
    try:
        import time
        inputs = tokenizer(request.prompt, return_tensors="pt").to(model.device)
        
        start_time = time.time()
        outputs = model.generate(
            **inputs,
            max_new_tokens=request.max_tokens,
            temperature=request.temperature,
            do_sample=request.temperature > 0
        )
        end_time = time.time()
        
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        inference_time = end_time - start_time
        tokens_per_second = request.max_tokens / inference_time if inference_time > 0 else 0
        
        return GenerateResponse(
            text=generated_text,
            inference_time=inference_time,
            tokens_per_second=tokens_per_second
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
async def health_check():
    return {"status": "healthy", "model_loaded": model is not None}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

### Тестирование API локально

In [ ]:
# Код для тестирования API (запускать после старта сервера в отдельном процессе)
import requests
import json

def test_api_endpoint(prompt: str, max_tokens: int = 50):
    """Тестирует API endpoint."""
    url = "http://localhost:8000/generate"
    
    payload = {
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": 0.7
    }
    
    try:
        response = requests.post(url, json=payload, timeout=60)
        response.raise_for_status()
        result = response.json()
        
        print(f"Запрос: {prompt}")
        print(f"Ответ: {result['text']}")
        print(f"Время: {result['inference_time']:.2f} сек")
        print(f"Скорость: {result['tokens_per_second']:.2f} токенов/сек")
        
        return result
    except requests.exceptions.RequestException as e:
        print(f"Ошибка подключения к API: {e}")
        print("Убедитесь, что сервер запущен командой: python llm_server.py")
        return None

# Пример использования (раскомментировать после запуска сервера)
# test_api_endpoint("Объясни простыми словами, что такое машинное обучение.")

### Создание Dockerfile для контейнеризации

In [ ]:
%%writefile Dockerfile

FROM python:3.10-slim

WORKDIR /app

# Установка системных зависимостей
RUN apt-get update && apt-get install -y \
    git \
    curl \
    && rm -rf /var/lib/apt/lists/*

# Установка Python зависимостей
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Копирование кода приложения
COPY llm_server.py .

# Экспорт порта
EXPOSE 8000

# Запуск приложения
CMD ["python", "llm_server.py"]

In [ ]:
%%writefile requirements.txt
transformers>=4.35.0
accelerate>=0.24.0
bitsandbytes>=0.41.0
torch>=2.0.0
fastapi>=0.104.0
uvicorn>=0.24.0
pydantic>=2.0.0

### Задание 1: Эксперименты с различными уровнями квантования

Сравните производительность и качество моделей с разными конфигурациями:
1. Без квантования (float16)
2. 8-битное квантование
3. 4-битное квантование

Измерьте:
- Время загрузки модели
- Использование памяти
- Скорость генерации (токенов/сек)
- Качество ответов (субъективная оценка)

In [ ]:
# Ваш код здесь
# Сравните различные конфигурации квантования

### Задание 2: Оптимизация параметров генерации

Исследуйте влияние параметров на скорость и качество:
- `max_new_tokens` — максимальное количество токенов
- `temperature` — температура выборки
- `top_p` / `top_k` — параметры выборки
- `do_sample` — сэмплирование vs жадная декодирование

In [ ]:
# Ваш код здесь
# Исследуйте влияние параметров генерации

### Задание 3: Батчинг запросов

Реализуйте обработку пакетных запросов для повышения пропускной способности API.

In [ ]:
# Ваш код здесь
# Реализуйте батчинг запросов

## Контрольные вопросы

1. Какие преимущества и недостатки у 4-битного квантования по сравнению с 8-битным?
2. Как работает PagedAttention в vLLM и почему он ускоряет инференс?
3. В чем разница между eager execution и графовыми оптимизациями?
4. Какие метрики важны для мониторинга продакшн LLM-сервиса?
5. Как масштабировать LLM-сервис для обработки тысяч запросов в секунду?

## Требования к отчету

1. Сравнительная таблица производительности различных конфигураций квантования
2. Графики зависимости скорости генерации от параметров
3. Описание реализованного API и примеры запросов
4. Анализ результатов экспериментов с батчингом
5. Ответы на контрольные вопросы
6. Выводы о最佳 практиках деплоя LLM